# AWP Fundamentals — No LLM Required

This notebook teaches the core concepts of the **Agent Workflow Protocol (AWP)** using only local parsing, validation, and model inspection. No API keys or LLM calls are needed.

## What is AWP?

AWP is an open standard for defining and orchestrating multi-agent workflows. It separates **workflow definition** (YAML) from **implementation** (Python), organized in **7 semantic layers**:

| Layer | Name | Purpose |
|-------|------|---------|
| 0 | **Manifest** | Workflow metadata, version, dependencies |
| 1 | **Identity** | Agent identity, role, description |
| 2 | **Capabilities** | Tools, skills, code execution |
| 3 | **Communication** | Inter-agent messaging, bus config |
| 4 | **Memory** | Short-term, long-term, semantic memory |
| 5 | **Orchestration** | DAG graph, delegation loop, execution config |
| 6 | **Observability** | Logging, metrics, tracing |

### Autonomy Levels (A0 - A4)

AWP defines an **autonomy spectrum** describing how self-directed a workflow is:

- **A0 Prescribed** — Static DAG, predefined agents, fixed execution order
- **A1 Adaptive** — Conditional execution, loops, fan-out, multi-agent DAG
- **A2 Delegating** — Manager spawns workers dynamically (delegation loop)
- **A3 Self-Tooling** — Agents create tools and skills at runtime
- **A4 Self-Organizing** — Recursive delegation with budget distribution

---

## 1. Setup and Installation

Install the AWP reference implementation (run once):

In [1]:
# Install AWP (editable mode from source)
# Uncomment and run if not already installed:
# !pip install -e ../../reference/python/

import awp
print(f"AWP version: {awp.__version__}")
print("AWP is installed and ready.")

AWP version: 1.0.0
AWP is installed and ready.


In [2]:
# Set working directory to the project root so all relative paths work
import os
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath("")).parent.parent.parent
# Fallback if the notebook is run from an unexpected location
if not (PROJECT_ROOT / "reference" / "python").exists():
    PROJECT_ROOT = Path("/home/shumway/projects/agent-workflow-protocol")

os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")
print(f"Examples dir exists: {(PROJECT_ROOT / 'examples').exists()}")

Working directory: /home/shumway/projects/agent-workflow-protocol
Examples dir exists: True


In [3]:
# Verify all key imports
from awp.parser import parse_manifest, parse_agent, resolve_templates
from awp.validator import (
    validate_schema,
    validate_graph,
    validate_contracts,
    check_compliance,
    validate_rules,
    AutonomyLevel,
)
from awp.models import (
    AWPManifest,
    AWPAgent,
    AWPOrchestrationConfig,
    GraphNode,
    DelegationLoopConfig,
    DelegationBudget,
    StateModel,
    MemoryConfig,
    CommunicationConfig,
    ObservabilityConfig,
    SecurityConfig,
)

print("All imports successful!")
print(f"AutonomyLevel members: {[l.name for l in AutonomyLevel]}")

All imports successful!
AutonomyLevel members: ['A0_PRESCRIBED', 'A1_ADAPTIVE', 'A2_DELEGATING', 'A3_SELF_TOOLING', 'A4_SELF_ORGANIZING']


---

## 2. Parsing Workflow YAML

Every AWP project has a `workflow.awp.yaml` file that defines the manifest (Layer 0) and orchestration graph (Layer 5). The `parse_manifest()` function reads this file and returns a validated `AWPManifest` Pydantic model.

### 2a. Hello World (A0 — single agent)

In [4]:
# Parse the simplest example: hello-world (A0 Prescribed)
hello_manifest = parse_manifest("examples/01-hello-world/workflow.awp.yaml")

print("=== Manifest Fields ===")
print(f"AWP version:    {hello_manifest.awp}")
print(f"Workflow name:  {hello_manifest.workflow.name}")
print(f"Version:        {hello_manifest.workflow.version}")
print(f"Description:    {hello_manifest.workflow.description}")
print(f"Tags:           {hello_manifest.workflow.tags}")
print(f"Author:         {hello_manifest.workflow.author}")

=== Manifest Fields ===
AWP version:    1.0.0
Workflow name:  hello-world
Version:        1.0.0
Description:    Simple greeting workflow demonstrating A0 Prescribed autonomy level
Tags:           ['example', 'a0', 'prescribed']
Author:         AWP Examples


In [5]:
# Access the orchestration config
orch = hello_manifest.orchestration
print(f"Engine:           {orch.engine}")
print(f"Execution mode:   {orch.execution.mode}")
print(f"Timeout (total):  {orch.execution.timeout.total}s")
print(f"Max parallel:     {orch.execution.max_parallel_agents}")
print()

# Inspect the graph nodes
print("=== Graph Nodes ===")
for node in orch.graph:
    print(f"  Node ID:      {node.id}")
    print(f"  Agent:        {node.agent}")
    print(f"  Depends on:   {node.depends_on}")
    print(f"  Share output: {node.share_output}")
    print()

Engine:           dag
Execution mode:   sequential
Timeout (total):  300s
Max parallel:     4

=== Graph Nodes ===
  Node ID:      greeter
  Agent:        greeter
  Depends on:   []
  Share output: ['greeting', 'tone']



### 2b. Research Pipeline (A1 — multi-agent DAG)

In [6]:
# Parse the research pipeline: 3 agents with dependencies
research_manifest = parse_manifest("examples/02-research-pipeline/workflow.awp.yaml")

print(f"Workflow: {research_manifest.workflow.name}")
print(f"Description: {research_manifest.workflow.description}")
print(f"Tags: {research_manifest.workflow.tags}")
print()

# Show the multi-agent graph
print("=== Multi-Agent DAG ===")
for node in research_manifest.orchestration.graph:
    deps = node.depends_on if node.depends_on else ["(root)"]
    print(f"  {node.id:15s} <- {deps}  | shares: {node.share_output}")

print()
print("Execution flow: planner -> researcher -> writer")
print(f"State sharing strategy: {research_manifest.state.sharing.strategy}")

Workflow: research-pipeline
Description: Multi-agent research pipeline demonstrating A1 Adaptive autonomy level with state sharing
Tags: ['example', 'a1', 'adaptive', 'research']

=== Multi-Agent DAG ===
  planner         <- ['(root)']  | shares: ['research_questions', 'search_strategy']
  researcher      <- ['planner']  | shares: ['findings', 'sources']
  writer          <- ['researcher']  | shares: ['report']

Execution flow: planner -> researcher -> writer
State sharing strategy: selective


---

## 3. Parsing Agent YAML

Each agent has its own `agent.awp.yaml` file defining its identity (Layer 1), model config, prompt architecture, and output contract. The `parse_agent()` function parses this into an `AWPAgent` model.

In [7]:
# Parse the greeter agent from hello-world
greeter = parse_agent("examples/01-hello-world/agents/greeter/agent.awp.yaml")

print("=== Agent Identity (Layer 1) ===")
print(f"  ID:          {greeter.identity.id}")
print(f"  Role:        {greeter.identity.role}")
print(f"  Description: {greeter.identity.description}")
print(f"  AWP Agent:   {greeter.awp_agent}")
print()

print("=== Model Config ===")
print(f"  Model name:  {greeter.model.name!r} (empty = resolved from LLM_MODEL env var)")
print(f"  Temperature: {greeter.model.parameters.temperature}")
print(f"  Max tokens:  {greeter.model.parameters.max_tokens}")
print()

print("=== Prompt Config ===")
print(f"  System prompt: {greeter.prompt.system}")
print(f"  User template: {greeter.prompt.user_template}")
print()

print("=== Output Contract ===")
print(f"  Format:     {greeter.output.format}")
print(f"  Validation: mode={greeter.output.validation.mode}, on_invalid={greeter.output.validation.on_invalid}")
print(f"  Fields:")
for field_name, field_spec in greeter.output.contract.items():
    print(f"    {field_name:15s} type={field_spec.type:8s} required={field_spec.required}")

=== Agent Identity (Layer 1) ===
  ID:          greeter
  Role:        greeting_specialist
  Description: Generates personalized greetings based on user input
  AWP Agent:   1.0.0

=== Model Config ===
  Model name:  '' (empty = resolved from LLM_MODEL env var)
  Temperature: 0.7
  Max tokens:  512

=== Prompt Config ===
  System prompt: workflow/instructions/SYSTEM_PROMPT.md
  User template: workflow/prompt/00_INTRO.md

=== Output Contract ===
  Format:     json
  Validation: mode=strict, on_invalid=retry
  Fields:
    greeting        type=string   required=True
    tone            type=string   required=True
    confidence      type=number   required=True


In [8]:
# Parse the planner agent from research-pipeline
planner = parse_agent("examples/02-research-pipeline/agents/planner/agent.awp.yaml")

print(f"Agent: {planner.identity.id} ({planner.identity.role})")
print(f"Description: {planner.identity.description}")
print(f"Output fields: {list(planner.output.contract.keys())}")
print(f"Capabilities: {planner.capabilities}")

Agent: planner (research_planner)
Description: Analyzes research topics and creates structured research plans
Output fields: ['research_questions', 'search_strategy', 'confidence']
Capabilities: None


---

## 4. Schema Validation

AWP requires each agent to have an `output_schema.json` file (a JSON Schema). The `validate_schema()` function checks it conforms to AWP rules:

- **R17**: Must include a `confidence` field (type `number`, 0.0-1.0)
- **R18**: Root type must be `"object"`
- Must have `properties` and `required` arrays

In [9]:
# Validate the greeter's output schema
schema_path = "examples/01-hello-world/agents/greeter/workflow/output_schema/output_schema.json"
result = validate_schema(schema_path)

print(f"Schema: {schema_path}")
print(f"Valid:    {result.valid}")
print(f"Errors:   {result.errors}")
print(f"Warnings: {result.warnings}")

Schema: examples/01-hello-world/agents/greeter/workflow/output_schema/output_schema.json
Valid:    True
Errors:   []
Warnings: []


In [10]:
# Validate another schema from a delegation loop example
schema_path_2 = "examples/08-delegation-loop/agents/manager/workflow/output_schema/output_schema.json"
result_2 = validate_schema(schema_path_2)

print(f"Schema: {schema_path_2}")
print(f"Valid:    {result_2.valid}")
print(f"Errors:   {result_2.errors}")
print(f"Warnings: {result_2.warnings}")

Schema: examples/08-delegation-loop/agents/manager/workflow/output_schema/output_schema.json
Valid:    True
Errors:   []
Warnings: []


In [11]:
# Show what happens with a BROKEN schema (missing confidence, wrong root type)
import json
import tempfile

broken_schema = {
    "type": "array",  # Wrong! Must be "object"
    "items": {"type": "string"},
    # Missing: properties, required, confidence field
}

with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    json.dump(broken_schema, f)
    broken_path = f.name

result_broken = validate_schema(broken_path)
print("=== Broken Schema Validation ===")
print(f"Valid:  {result_broken.valid}")
print(f"Errors:")
for e in result_broken.errors:
    print(f"  - {e}")

os.unlink(broken_path)

=== Broken Schema Validation ===
Valid:  False
Errors:
  - R18: Root schema type must be 'object'
  - Schema must have at least one property
  - R17: Schema must include a 'confidence' field
  - R17: 'confidence' must be in 'required' array


---

## 5. Graph Validation

The `validate_graph()` function checks the orchestration DAG for structural correctness:

- **R2**: Unique agent IDs (snake_case)
- **R6**: All `depends_on` references must point to existing agents
- **R7**: No cycles in the graph (unless within declared loops)
- **Orphan detection**: All agents must be reachable from root nodes

In [12]:
# Validate the hello-world graph (trivial: 1 node, no deps)
result_hw = validate_graph(hello_manifest.orchestration)
print("=== Hello World Graph ===")
print(f"Valid:    {result_hw.valid}")
print(f"Errors:   {result_hw.errors}")
print(f"Warnings: {result_hw.warnings}")

=== Hello World Graph ===
Valid:    True
Errors:   []
Warnings: []


In [13]:
# Validate the research pipeline graph (3 nodes, linear chain)
result_rp = validate_graph(research_manifest.orchestration)
print("=== Research Pipeline Graph ===")
print(f"Valid:    {result_rp.valid}")
print(f"Errors:   {result_rp.errors}")
print(f"Warnings: {result_rp.warnings}")

=== Research Pipeline Graph ===
Valid:    True
Errors:   []
Warnings: []


In [14]:
# Build a BROKEN graph with a cycle to demonstrate error detection
broken_orch = AWPOrchestrationConfig(
    engine="dag",
    graph=[
        GraphNode(id="alpha", agent="alpha", depends_on=["gamma"]),  # alpha depends on gamma
        GraphNode(id="beta", agent="beta", depends_on=["alpha"]),    # beta depends on alpha
        GraphNode(id="gamma", agent="gamma", depends_on=["beta"]),   # gamma depends on beta -> CYCLE!
    ],
)

result_cycle = validate_graph(broken_orch)
print("=== Broken Graph (Cycle) ===")
print(f"Valid:  {result_cycle.valid}")
print(f"Errors:")
for e in result_cycle.errors:
    print(f"  - {e}")

=== Broken Graph (Cycle) ===
Valid:  False
Errors:
  - R7: Cycle detected involving agents: ['alpha', 'beta', 'gamma']


In [15]:
# Build a graph with a dangling dependency (R6 violation)
broken_orch_r6 = AWPOrchestrationConfig(
    engine="dag",
    graph=[
        GraphNode(id="step_one", agent="step_one", depends_on=[]),
        GraphNode(id="step_two", agent="step_two", depends_on=["nonexistent_agent"]),
    ],
)

result_r6 = validate_graph(broken_orch_r6)
print("=== Broken Graph (Dangling Dependency) ===")
print(f"Valid:  {result_r6.valid}")
for e in result_r6.errors:
    print(f"  - {e}")

=== Broken Graph (Dangling Dependency) ===
Valid:  False
  - R6: Agent 'step_two' depends on 'nonexistent_agent' which doesn't exist
  - R7: Cycle detected involving agents: ['step_two']


In [16]:
# Build a graph with a duplicate ID (R2 violation)
broken_orch_r2 = AWPOrchestrationConfig(
    engine="dag",
    graph=[
        GraphNode(id="analyzer", agent="analyzer_v1", depends_on=[]),
        GraphNode(id="analyzer", agent="analyzer_v2", depends_on=[]),  # Duplicate ID!
    ],
)

result_r2 = validate_graph(broken_orch_r2)
print("=== Broken Graph (Duplicate ID) ===")
print(f"Valid:  {result_r2.valid}")
for e in result_r2.errors:
    print(f"  - {e}")

=== Broken Graph (Duplicate ID) ===
Valid:  False
  - R2: Duplicate agent ID: 'analyzer'


---

## 6. Compliance / Autonomy Level Checking

The `check_compliance()` function analyzes a workflow manifest and its agents to determine the achieved autonomy level (A0-A4). It checks cross-cutting concerns (security, observability) and level-specific requirements.

Key parameters:
- `manifest`: The parsed `AWPManifest`
- `agents`: A dict of `agent_id -> AWPAgent` (parsed agent configs)
- `target_level`: The autonomy level to check up to (an `AutonomyLevel` enum value)
- `workflow_path`: Path to the workflow directory

In [17]:
# Helper function to load agents for a workflow
def load_agents(workflow_dir: Path) -> dict[str, "AWPAgent"]:
    """Load all agent.awp.yaml files from a workflow's agents/ directory."""
    agents = {}
    agents_dir = workflow_dir / "agents"
    if not agents_dir.exists():
        return agents
    for agent_dir in sorted(agents_dir.iterdir()):
        agent_yaml = agent_dir / "agent.awp.yaml"
        if agent_yaml.exists():
            try:
                agent = parse_agent(agent_yaml)
                agents[agent.identity.id] = agent
            except Exception as e:
                print(f"  Warning: Could not parse {agent_yaml}: {e}")
    return agents


def show_compliance(name: str, result):
    """Pretty-print a ComplianceResult."""
    print(f"=== {name} ===")
    print(f"  Achieved level: {result.level.name} ({result.level_name})")
    print(f"  Max achievable: {result.max_achievable.name}")
    print(f"  Compliant:      {result.compliant}")
    if result.cross_cutting:
        print(f"  Cross-cutting:  {result.cross_cutting}")
    if result.checks:
        print(f"  Checks ({len(result.checks)}):")
        for k, v in result.checks.items():
            status = "PASS" if v else "FAIL"
            print(f"    [{status}] {k}")
    if result.errors:
        print(f"  Errors:")
        for e in result.errors:
            print(f"    - {e}")
    if result.warnings:
        print(f"  Warnings:")
        for w in result.warnings:
            print(f"    - {w}")
    print()

print("Helper functions defined.")

Helper functions defined.


In [18]:
# Example 01: Hello World (A0 Prescribed)
hw_dir = Path("examples/01-hello-world")
hw_manifest = parse_manifest(hw_dir / "workflow.awp.yaml")
hw_agents = load_agents(hw_dir)
print(f"Loaded agents: {list(hw_agents.keys())}")

hw_compliance = check_compliance(
    hw_manifest, hw_agents,
    workflow_path=hw_dir,
    target_level=AutonomyLevel.A4_SELF_ORGANIZING,
)
show_compliance("01 Hello World", hw_compliance)

Loaded agents: ['greeter']
=== 01 Hello World ===
  Achieved level: A0_PRESCRIBED (Prescribed)
  Max achievable: A4_SELF_ORGANIZING
  Compliant:      False
  Cross-cutting:  {'security_configured': False, 'observability_configured': False}
  Checks (14):
    [PASS] manifest_present
    [PASS] awp_version_set
    [PASS] workflow_name_valid
    [PASS] at_least_one_agent
    [PASS] agent_greeter_has_contract
    [PASS] has_orchestration
    [PASS] has_graph_or_delegation
    [FAIL] has_adaptive_features
    [FAIL] multi_agent_dag
    [FAIL] delegation_loop_engine
    [FAIL] delegation_loop_config
    [FAIL] dynamic_tools_enabled
    [FAIL] has_tool_creator_agent
    [FAIL] delegation_allows_tool_creation
  Errors:
    - A1: Requires adaptive features (when, loop, fan_out) or multi-agent DAG
    - A2: Requires engine=delegation_loop with delegation_loop config
    - A3: Requires dynamic_tools.enabled, agent with tool_creation, or delegation loop with tool_creation
    - A4: Delegation loop

In [19]:
# Example 02: Research Pipeline (A1 Adaptive)
rp_dir = Path("examples/02-research-pipeline")
rp_manifest = parse_manifest(rp_dir / "workflow.awp.yaml")
rp_agents = load_agents(rp_dir)
print(f"Loaded agents: {list(rp_agents.keys())}")

rp_compliance = check_compliance(
    rp_manifest, rp_agents,
    workflow_path=rp_dir,
    target_level=AutonomyLevel.A4_SELF_ORGANIZING,
)
show_compliance("02 Research Pipeline", rp_compliance)

Loaded agents: ['planner', 'researcher', 'writer']
=== 02 Research Pipeline ===
  Achieved level: A1_ADAPTIVE (Adaptive)
  Max achievable: A4_SELF_ORGANIZING
  Compliant:      False
  Cross-cutting:  {'security_configured': False, 'observability_configured': False}
  Checks (16):
    [PASS] manifest_present
    [PASS] awp_version_set
    [PASS] workflow_name_valid
    [PASS] at_least_one_agent
    [PASS] agent_planner_has_contract
    [PASS] agent_researcher_has_contract
    [PASS] agent_writer_has_contract
    [PASS] has_orchestration
    [PASS] has_graph_or_delegation
    [FAIL] has_adaptive_features
    [PASS] multi_agent_dag
    [FAIL] delegation_loop_engine
    [FAIL] delegation_loop_config
    [FAIL] dynamic_tools_enabled
    [FAIL] has_tool_creator_agent
    [FAIL] delegation_allows_tool_creation
  Errors:
    - A2: Requires engine=delegation_loop with delegation_loop config
    - A3: Requires dynamic_tools.enabled, agent with tool_creation, or delegation loop with tool_creation

In [20]:
# Example 08: Delegation Loop (A2 Delegating)
dl_dir = Path("examples/08-delegation-loop")
dl_manifest = parse_manifest(dl_dir / "workflow.awp.yaml")
dl_agents = load_agents(dl_dir)
print(f"Loaded agents: {list(dl_agents.keys())}")

dl_compliance = check_compliance(
    dl_manifest, dl_agents,
    workflow_path=dl_dir,
    target_level=AutonomyLevel.A4_SELF_ORGANIZING,
)
show_compliance("08 Delegation Loop", dl_compliance)

Loaded agents: ['manager']
=== 08 Delegation Loop ===
  Achieved level: A3_SELF_TOOLING (Self-Tooling)
  Max achievable: A4_SELF_ORGANIZING
  Compliant:      False
  Cross-cutting:  {'security_configured': False, 'observability_configured': False}
  Checks (17):
    [PASS] manifest_present
    [PASS] awp_version_set
    [PASS] workflow_name_valid
    [PASS] at_least_one_agent
    [PASS] agent_manager_has_contract
    [PASS] has_orchestration
    [PASS] has_graph_or_delegation
    [PASS] has_adaptive_features
    [PASS] delegation_loop_engine
    [PASS] delegation_loop_config
    [PASS] has_budget
    [FAIL] dynamic_tools_enabled
    [FAIL] has_tool_creator_agent
    [PASS] delegation_allows_tool_creation
    [PASS] has_safety_envelope
    [PASS] allows_recursive_delegation
    [FAIL] observability_required_a4
  Errors:
    - A4: Observability required for self-organizing workflows
  Warnings:
    - Security config recommended for all workflows
    - Observability config recommended for

In [21]:
# Example 09: Recursive Delegation (A2+)
rd_dir = Path("examples/09-recursive-delegation")
rd_manifest = parse_manifest(rd_dir / "workflow.awp.yaml")
rd_agents = load_agents(rd_dir)
print(f"Loaded agents: {list(rd_agents.keys())}")

rd_compliance = check_compliance(
    rd_manifest, rd_agents,
    workflow_path=rd_dir,
    target_level=AutonomyLevel.A4_SELF_ORGANIZING,
)
show_compliance("09 Recursive Delegation", rd_compliance)

Loaded agents: ['analyzer']
=== 09 Recursive Delegation ===
  Achieved level: A3_SELF_TOOLING (Self-Tooling)
  Max achievable: A4_SELF_ORGANIZING
  Compliant:      False
  Cross-cutting:  {'security_configured': False, 'observability_configured': False}
  Checks (17):
    [PASS] manifest_present
    [PASS] awp_version_set
    [PASS] workflow_name_valid
    [PASS] at_least_one_agent
    [PASS] agent_analyzer_has_contract
    [PASS] has_orchestration
    [PASS] has_graph_or_delegation
    [PASS] has_adaptive_features
    [PASS] delegation_loop_engine
    [PASS] delegation_loop_config
    [PASS] has_budget
    [FAIL] dynamic_tools_enabled
    [FAIL] has_tool_creator_agent
    [PASS] delegation_allows_tool_creation
    [PASS] has_safety_envelope
    [PASS] allows_recursive_delegation
    [FAIL] observability_required_a4
  Errors:
    - A4: Observability required for self-organizing workflows
  Warnings:
    - Security config recommended for all workflows
    - Observability config recommen

In [22]:
# Example 12: Full Autonomy Test (A3/A4)
fa_dir = Path("examples/12-full-autonomy-test")
fa_manifest = parse_manifest(fa_dir / "workflow.awp.yaml")
fa_agents = load_agents(fa_dir)
print(f"Loaded agents: {list(fa_agents.keys())}")

fa_compliance = check_compliance(
    fa_manifest, fa_agents,
    workflow_path=fa_dir,
    target_level=AutonomyLevel.A4_SELF_ORGANIZING,
)
show_compliance("12 Full Autonomy Test", fa_compliance)

Loaded agents: ['manager']


=== 12 Full Autonomy Test ===
  Achieved level: A3_SELF_TOOLING (Self-Tooling)
  Max achievable: A4_SELF_ORGANIZING
  Compliant:      False
  Cross-cutting:  {'security_configured': False, 'observability_configured': False}
  Checks (17):
    [PASS] manifest_present
    [PASS] awp_version_set
    [PASS] workflow_name_valid
    [PASS] at_least_one_agent
    [PASS] agent_manager_has_contract
    [PASS] has_orchestration
    [PASS] has_graph_or_delegation
    [PASS] has_adaptive_features
    [PASS] delegation_loop_engine
    [PASS] delegation_loop_config
    [PASS] has_budget
    [PASS] dynamic_tools_enabled
    [FAIL] has_tool_creator_agent
    [PASS] delegation_allows_tool_creation
    [PASS] has_safety_envelope
    [PASS] allows_recursive_delegation
    [FAIL] observability_required_a4
  Errors:
    - A4: Observability required for self-organizing workflows
  Warnings:
    - Security config recommended for all workflows
    - Observability config recommended for all workflows



In [23]:
# Summary comparison table
print("=== Autonomy Level Summary ===")
print(f"{'Example':<35} {'Achieved':<25} {'Level Name'}")
print("-" * 75)
for name, result in [
    ("01 Hello World", hw_compliance),
    ("02 Research Pipeline", rp_compliance),
    ("08 Delegation Loop", dl_compliance),
    ("09 Recursive Delegation", rd_compliance),
    ("12 Full Autonomy Test", fa_compliance),
]:
    print(f"  {name:<33} {result.level.name:<23} {result.level_name}")

=== Autonomy Level Summary ===
Example                             Achieved                  Level Name
---------------------------------------------------------------------------
  01 Hello World                    A0_PRESCRIBED           Prescribed
  02 Research Pipeline              A1_ADAPTIVE             Adaptive
  08 Delegation Loop                A3_SELF_TOOLING         Self-Tooling
  09 Recursive Delegation           A3_SELF_TOOLING         Self-Tooling
  12 Full Autonomy Test             A3_SELF_TOOLING         Self-Tooling


---

## 7. Rule Validation (R1-R26)

The `validate_rules()` function checks all 26 AWP rules against a parsed workflow. These cover naming conventions, graph structure, tool namespaces, output contracts, and more.

Key rules:
- **R1**: `workflow.name` must match the directory name
- **R2**: Unique agent IDs in snake_case
- **R3**: Agent `class_name` must be `"Agent"`
- **R5/R9**: Agent directories must exist
- **R6**: `depends_on` references must be valid
- **R11**: Required files in agent directories
- **R15**: Tool namespace collision with reserved namespaces
- **R17**: Output contract must include `confidence` field
- **R25/R26**: Dynamic tool namespace and code mode rules

Note: `validate_rules()` takes a `workflow_path` that is a `Path` pointing to the **workflow directory** (not the YAML file).

In [24]:
# Validate rules on the hello-world example
hw_rules = validate_rules(
    hw_manifest,
    hw_agents,
    workflow_path=Path("examples/01-hello-world"),
)

print("=== Rule Validation: Hello World ===")
print(f"Valid:  {hw_rules.valid}")
if hw_rules.errors:
    print("Errors:")
    for e in hw_rules.errors:
        print(f"  - {e}")
else:
    print("No errors -- all rules pass!")
if hw_rules.warnings:
    print("Warnings:")
    for w in hw_rules.warnings:
        print(f"  - {w}")

=== Rule Validation: Hello World ===
Valid:  True
No errors -- all rules pass!


In [25]:
# Validate rules on the research pipeline
rp_rules = validate_rules(
    rp_manifest,
    rp_agents,
    workflow_path=Path("examples/02-research-pipeline"),
)

print("=== Rule Validation: Research Pipeline ===")
print(f"Valid:  {rp_rules.valid}")
if rp_rules.errors:
    print("Errors:")
    for e in rp_rules.errors:
        print(f"  - {e}")
else:
    print("No errors -- all rules pass!")
if rp_rules.warnings:
    print("Warnings:")
    for w in rp_rules.warnings:
        print(f"  - {w}")

=== Rule Validation: Research Pipeline ===
Valid:  True
No errors -- all rules pass!


In [26]:
# Validate the delegation loop example
dl_rules = validate_rules(
    dl_manifest,
    dl_agents,
    workflow_path=Path("examples/08-delegation-loop"),
)

print("=== Rule Validation: Delegation Loop ===")
print(f"Valid:  {dl_rules.valid}")
if dl_rules.errors:
    print(f"Errors ({len(dl_rules.errors)}):")
    for e in dl_rules.errors:
        print(f"  - {e}")
else:
    print("No errors -- all rules pass!")

=== Rule Validation: Delegation Loop ===
Valid:  True
No errors -- all rules pass!


---

## 8. Pydantic Models Deep Dive

All AWP structures are Pydantic models, which means they have:
- Automatic validation on construction
- Type coercion where possible
- Easy serialization to dict and JSON
- Rich field metadata

Let's explore creating models programmatically.

In [27]:
# Create a GraphNode programmatically
node = GraphNode(
    id="data_processor",
    agent="data_processor",
    depends_on=["data_loader"],
    share_output=["processed_data", "statistics"],
    description="Processes raw data and computes statistics",
    on_failure="abort",
    retry=2,
    timeout=60,
)

print("=== GraphNode ===")
print(f"  ID:           {node.id}")
print(f"  Agent:        {node.agent}")
print(f"  Depends on:   {node.depends_on}")
print(f"  Share output: {node.share_output}")
print(f"  On failure:   {node.on_failure}")
print(f"  Retry:        {node.retry}")
print(f"  Timeout:      {node.timeout}")
print()

# Serialize to dict
node_dict = node.model_dump()
print("As dict (first 5 keys):")
for k in list(node_dict.keys())[:5]:
    print(f"  {k}: {node_dict[k]}")

=== GraphNode ===
  ID:           data_processor
  Agent:        data_processor
  Depends on:   ['data_loader']
  Share output: ['processed_data', 'statistics']
  On failure:   abort
  Retry:        2
  Timeout:      60

As dict (first 5 keys):
  id: data_processor
  agent: data_processor
  enabled: True
  depends_on: ['data_loader']
  share_input: {}


In [28]:
# Create a DelegationBudget
budget = DelegationBudget(
    max_loops=10,
    max_total_workers=20,
    max_total_tokens=500_000,
    max_wall_time=300,
    max_tool_calls=100,
    max_depth=3,
)

print("=== DelegationBudget ===")
print(f"  Max loops:    {budget.max_loops}")
print(f"  Max workers:  {budget.max_total_workers}")
print(f"  Max tokens:   {budget.max_total_tokens:,}")
print(f"  Max wall:     {budget.max_wall_time}s")
print(f"  Max tools:    {budget.max_tool_calls}")
print(f"  Max depth:    {budget.max_depth}")
print()

# Serialize to JSON string
print("As JSON:")
print(budget.model_dump_json(indent=2))

=== DelegationBudget ===
  Max loops:    10
  Max workers:  20
  Max tokens:   500,000
  Max wall:     300s
  Max tools:    100
  Max depth:    3

As JSON:
{
  "max_loops": 10,
  "max_total_workers": 20,
  "max_total_tokens": 500000,
  "max_wall_time": 300,
  "max_tool_calls": 100,
  "max_depth": 3
}


In [29]:
# Create a full DelegationLoopConfig
dl_config = DelegationLoopConfig(
    manager="agents/manager",
    budget=DelegationBudget(
        max_loops=5,
        max_total_workers=10,
        max_total_tokens=200_000,
        max_wall_time=120,
        max_depth=2,
    ),
)

print("=== DelegationLoopConfig ===")
print(f"  Manager:           {dl_config.manager}")
print(f"  Budget max loops:  {dl_config.budget.max_loops}")
print(f"  Budget max depth:  {dl_config.budget.max_depth}")
print(f"  Stall detection:   {dl_config.termination.enabled}")
print(f"  Validation (det.): {dl_config.validation.deterministic.always}")
print(f"  Validation (LLM):  {dl_config.validation.llm.enabled}")
print(f"  Forbidden tools:   {dl_config.worker_policy.enforced.forbidden_tools}")

=== DelegationLoopConfig ===
  Manager:           agents/manager
  Budget max loops:  5
  Budget max depth:  2
  Stall detection:   True
  Validation (det.): True
  Validation (LLM):  True
  Forbidden tools:   ['file.write_outside_workspace', 'shell.execute']


In [30]:
# Create a complete AWPOrchestrationConfig with a DAG graph
orch_config = AWPOrchestrationConfig(
    engine="dag",
    graph=[
        GraphNode(id="fetcher", agent="fetcher", share_output=["raw_data"]),
        GraphNode(id="analyzer", agent="analyzer", depends_on=["fetcher"], share_output=["analysis"]),
        GraphNode(id="reporter", agent="reporter", depends_on=["analyzer"], share_output=["report"]),
    ],
)

print("=== Custom Orchestration Config ===")
print(f"  Engine: {orch_config.engine}")
print(f"  Nodes:  {len(orch_config.graph)}")
for n in orch_config.graph:
    print(f"    {n.id} -> depends_on={n.depends_on}, shares={n.share_output}")
print()

# Validate it!
result = validate_graph(orch_config)
print(f"  Graph valid: {result.valid}")

=== Custom Orchestration Config ===
  Engine: dag
  Nodes:  3
    fetcher -> depends_on=[], shares=['raw_data']
    analyzer -> depends_on=['fetcher'], shares=['analysis']
    reporter -> depends_on=['analyzer'], shares=['report']

  Graph valid: True


In [31]:
# Show Pydantic validation in action: invalid values are caught
from pydantic import ValidationError

# Try creating an AgentId with invalid characters
try:
    bad_node = GraphNode(id="InvalidCamelCase", agent="test")
except ValidationError as e:
    print("=== Pydantic Validation Error ===")
    print(f"Error creating GraphNode with id='InvalidCamelCase':")
    for err in e.errors():
        print(f"  Field: {err['loc']}")
        print(f"  Error: {err['msg']}")

print()

# Try creating a manifest with invalid SemVer
try:
    bad_manifest = AWPManifest(
        awp="not-a-version",
        workflow={"name": "test", "version": "1.0.0", "description": "test"},
    )
except ValidationError as e:
    print("Error creating AWPManifest with awp='not-a-version':")
    for err in e.errors():
        print(f"  Field: {err['loc']}")
        print(f"  Error: {err['msg']}")

=== Pydantic Validation Error ===
Error creating GraphNode with id='InvalidCamelCase':
  Field: ('id',)
  Error: Value error, Invalid AgentId: InvalidCamelCase

Error creating AWPManifest with awp='not-a-version':
  Field: ('awp',)
  Error: Value error, Invalid SemVer: not-a-version


In [32]:
# Explore model field names and types
print("=== AWPManifest Fields ===")
for name, field_info in AWPManifest.model_fields.items():
    annotation = field_info.annotation
    required = field_info.is_required()
    print(f"  {name:20s} type={str(annotation):40s} required={required}")

print()
print("=== GraphNode Fields ===")
for name, field_info in GraphNode.model_fields.items():
    annotation = field_info.annotation
    default = field_info.default
    print(f"  {name:15s} type={str(annotation):45s} default={default}")

=== AWPManifest Fields ===
  awp                  type=<class 'str'>                            required=True
  workflow             type=<class 'awp.models.manifest.WorkflowMetadata'> required=True
  orchestration        type=typing.Optional[typing.Any]              required=False
  state                type=typing.Optional[typing.Any]              required=False
  memory               type=typing.Optional[typing.Any]              required=False
  communication        type=typing.Optional[typing.Any]              required=False
  observability        type=typing.Optional[typing.Any]              required=False
  custom_tools         type=typing.Optional[typing.Any]              required=False
  dynamic_tools        type=typing.Optional[awp.models.manifest.DynamicToolsConfig] required=False
  security             type=typing.Optional[typing.Any]              required=False
  env                  type=typing.Optional[typing.Any]              required=False
  settings             type=ty

In [33]:
# Inspect the StateModel
state = hello_manifest.state
print("=== StateModel from Hello World ===")
print(f"  Model:    {state.model}")
print(f"  Sharing:  strategy={state.sharing.strategy}")
print()

# Create one from scratch
custom_state = StateModel(
    model="shared_dict",
    sharing={"strategy": "selective"},
)
print(f"Custom state: model={custom_state.model}, strategy={custom_state.sharing.strategy}")

=== StateModel from Hello World ===
  Model:    shared_dict
  Sharing:  strategy=full

Custom state: model=shared_dict, strategy=selective


---

## 9. Summary and Cheat Sheet

### Parsing Functions

| Function | Input | Returns |
|----------|-------|---------|
| `parse_manifest(path)` | Path to `workflow.awp.yaml` | `AWPManifest` |
| `parse_agent(path)` | Path to `agent.awp.yaml` | `AWPAgent` |
| `resolve_templates(data, context)` | Raw YAML dict + context | Resolved dict |

### Validation Functions

| Function | Input | Returns |
|----------|-------|---------|
| `validate_schema(path)` | Path to `output_schema.json` | `ValidationResult(valid, errors, warnings)` |
| `validate_graph(orch_config)` | `AWPOrchestrationConfig` | `ValidationResult` |
| `validate_rules(manifest, agents, workflow_path)` | Manifest + agents dict + Path to workflow dir | `ValidationResult` |
| `check_compliance(manifest, agents, workflow_path, target_level)` | Manifest + agents + path + `AutonomyLevel` enum | `ComplianceResult` |
| `validate_contracts(manifest, agents)` | Manifest + agents dict | `ValidationResult` |

### Key Models

| Model | Layer | Purpose |
|-------|-------|---------|
| `AWPManifest` | 0 | Root workflow document |
| `AWPAgent` | 1 | Agent identity, model, prompt, output contract |
| `AWPOrchestrationConfig` | 5 | DAG graph + execution config |
| `GraphNode` | 5 | Single node in the DAG |
| `DelegationLoopConfig` | 5 | Manager-worker loop configuration |
| `DelegationBudget` | 5 | Resource limits for delegation loops |
| `StateModel` | - | State sharing configuration |
| `SecurityConfig` | - | Security constraints |
| `ObservabilityConfig` | 6 | Logging, metrics, tracing |

### Autonomy Levels

| Level | Name | Key Requirement |
|-------|------|-----------------|
| `AutonomyLevel.A0_PRESCRIBED` | Prescribed | Manifest + 1 agent + output contract |
| `AutonomyLevel.A1_ADAPTIVE` | Adaptive | Multi-agent DAG or conditional features |
| `AutonomyLevel.A2_DELEGATING` | Delegating | Delegation loop engine + budget |
| `AutonomyLevel.A3_SELF_TOOLING` | Self-Tooling | Dynamic tool creation + safety envelope |
| `AutonomyLevel.A4_SELF_ORGANIZING` | Self-Organizing | Recursive delegation (max_depth > 1) + observability |

In [34]:
# Quick reference: all imports in one place
print("""Quick Reference — Copy-Paste Imports:

from awp.parser import parse_manifest, parse_agent, resolve_templates
from awp.validator import (
    validate_schema, validate_graph, validate_contracts,
    check_compliance, validate_rules, AutonomyLevel,
)
from awp.models import (
    AWPManifest, AWPAgent, AWPOrchestrationConfig, GraphNode,
    DelegationLoopConfig, DelegationBudget, StateModel,
    MemoryConfig, CommunicationConfig, ObservabilityConfig,
    SecurityConfig,
)
""")
print("Notebook complete. All examples run locally without any LLM calls.")

Quick Reference — Copy-Paste Imports:

from awp.parser import parse_manifest, parse_agent, resolve_templates
from awp.validator import (
    validate_schema, validate_graph, validate_contracts,
    check_compliance, validate_rules, AutonomyLevel,
)
from awp.models import (
    AWPManifest, AWPAgent, AWPOrchestrationConfig, GraphNode,
    DelegationLoopConfig, DelegationBudget, StateModel,
    MemoryConfig, CommunicationConfig, ObservabilityConfig,
    SecurityConfig,
)

Notebook complete. All examples run locally without any LLM calls.
